# Basic Experiment Notebook
Motor imagery classification on PhysioNet EEG Motor Movement/Imagery Dataset.


## 1. Imports & Config

In [9]:
import os
import numpy as np
import mne
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import warnings

warnings.filterwarnings('ignore', message='Limited .* annotation.* outside the data range')
mne.set_log_level('WARNING')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

# ── Paths & dataset constants ────────────────────────────────────────────────
DATA_DIR = os.path.join(os.getcwd(), '../files')

# Runs with left/right hand motor imagery (Task 1: close left or right fist)
RUNS = ['R03', 'R04', 'R07', 'R08', 'R11', 'R12']

# Number of subjects to use (1–109); set to None to use all
N_SUBJECTS = 109

SFREQ     = 160    # Hz — native sample rate
TMIN      = 0.0   # epoch start (s) relative to event onset
TMAX      = 2.0   # epoch end   (s)
N_SAMPLES = int(SFREQ * (TMAX - TMIN))  # 320 samples per epoch
N_CHANNELS = 64

# ── Filtering ────────────────────────────────────────────────────────────────
NOTCH_FREQ  = 60.0 
BP_LOW      = 4.0         # bandpass lower edge  (Hz)
BP_HIGH     = 40.0        # bandpass upper edge  (Hz)

# ── Train / val / test split fractions ──────────────────────────────────────
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
# TEST_FRAC  = 1 - TRAIN_FRAC - VAL_FRAC  (implicit)

BATCH_SIZE = 64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cpu


## 2. Data Loading & Preprocessing

In [10]:
def load_subject(subject_id):
    """Load all runs for one subject, apply notch + bandpass, return (X, y).

    Returns
    -------
    X : ndarray, shape (n_epochs, n_channels, n_samples)
    y : ndarray, shape (n_epochs,)  — 0=left hand, 1=right hand
    """
    subject = f'S{subject_id:03d}'
    raws = []

    for run in RUNS:
        path = os.path.join(DATA_DIR, subject, f'{subject}{run}.edf')
        if not os.path.exists(path):
            continue
        raw = mne.io.read_raw_edf(path, preload=True, verbose=False)

        # Notch filter — remove power-line artifact
        # raw.notch_filter(NOTCH_FREQ, verbose=False)

        # Bandpass filter — keep motor-imagery relevant frequencies
        # raw.filter(BP_LOW, BP_HIGH, verbose=False)

        raws.append(raw)

    if not raws:
        return np.empty((0, N_CHANNELS, 0)), np.empty((0,), dtype=int)

    raw = mne.concatenate_raws(raws)
    events, event_id = mne.events_from_annotations(raw)

    # Some files only have T0 (rest); skip if T1/T2 not present
    if 'T1' not in event_id or 'T2' not in event_id:
        return np.empty((0, N_CHANNELS, 0)), np.empty((0,), dtype=int)

    epochs = mne.Epochs(
        raw,
        events,
        event_id={'T1': event_id['T1'], 'T2': event_id['T2']},
        tmin=TMIN, tmax=TMAX,
        baseline=None,
        preload=True,
        verbose=False,
    )

    X = epochs.get_data()  # (n_epochs, 64, n_times)
    y = epochs.events[:, -1]
    y = (y == event_id['T2']).astype(int)  # T1=left=0, T2=right=1

    # Trim / pad to exactly N_SAMPLES
    X_fixed = []
    for x in X:
        if x.shape[1] >= N_SAMPLES:
            x = x[:, :N_SAMPLES]
        else:
            x = np.pad(x, ((0, 0), (0, N_SAMPLES - x.shape[1])))
        X_fixed.append(x)

    return np.stack(X_fixed), y


def load_all_subjects(n_subjects=N_SUBJECTS):
    """Load epochs from all subjects and pool them into one array."""
    X_all, y_all = [], []
    for sid in range(1, n_subjects + 1):
        X, y = load_subject(sid)
        if len(X) == 0:
            continue
        X_all.append(X)
        y_all.append(y)
    return np.concatenate(X_all), np.concatenate(y_all)

In [11]:
print('Loading all subjects...')
X_all, y_all = load_all_subjects()
print(f'Total epochs: {len(X_all)} | Shape: {X_all.shape}')
print(f'Class balance — left: {(y_all==0).sum()} | right: {(y_all==1).sum()}')

Loading all subjects...
Total epochs: 9845 | Shape: (9845, 64, 320)
Class balance — left: 4951 | right: 4894


## 3. Normalisation & Random Train / Val / Test Split

In [12]:
def normalize(X):
    """Per-epoch, per-channel z-score normalisation + add channel dim for Conv2d."""
    mu  = X.mean(axis=2, keepdims=True)   # (N, C, 1)
    std = X.std(axis=2,  keepdims=True) + 1e-6
    X   = (X - mu) / std
    return X[:, np.newaxis, :, :]         # (N, 1, C, T)


class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [13]:
# Normalise all data first, then split
X_norm = normalize(X_all)

n_total = len(X_norm)
idx = np.random.permutation(n_total)

n_train = int(n_total * TRAIN_FRAC)
n_val   = int(n_total * VAL_FRAC)

train_idx = idx[:n_train]
val_idx   = idx[n_train : n_train + n_val]
test_idx  = idx[n_train + n_val :]

full_dataset = EEGDataset(X_norm, y_all)

train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(Subset(full_dataset, val_idx),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(Subset(full_dataset, test_idx),  batch_size=BATCH_SIZE)

print(f'Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')

Train: 6891 | Val: 1476 | Test: 1478


## 4. EEGNet with Inception Model

In [ ]:
class EEGNetInception(nn.Module):
    """EEGNet backbone with an EEG-ITNet-style inception front end.

    Input: (B, 1, n_channels, n_samples)
    """
    def __init__(self, n_channels=64, n_samples=320, n_classes=2,
                 inception_filters=(2, 4, 8), kernel_sizes=(16, 32, 64),
                 D=2, F2=16, dropout=0.25):
        super().__init__()

        if len(inception_filters) != len(kernel_sizes):
            raise ValueError('inception_filters and kernel_sizes must have the same length')

        self.temporal_branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(1, out_channels, (1, kernel_size), padding='same', bias=False),
                nn.BatchNorm2d(out_channels),
            )
            for out_channels, kernel_size in zip(inception_filters, kernel_sizes)
        ])

        temporal_channels = sum(inception_filters)

        # EEGNet-style depthwise spatial filtering after inception fusion.
        self.block1 = nn.Sequential(
            nn.Conv2d(
                temporal_channels,
                temporal_channels * D,
                (n_channels, 1),
                groups=temporal_channels,
                bias=False,
            ),
            nn.BatchNorm2d(temporal_channels * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )

        # EEGNet-style separable temporal refinement.
        self.block2 = nn.Sequential(
            nn.Conv2d(
                temporal_channels * D,
                temporal_channels * D,
                (1, 16),
                padding='same',
                groups=temporal_channels * D,
                bias=False,
            ),
            nn.Conv2d(temporal_channels * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )

        flatten_dim = self._infer_flatten_dim(n_channels, n_samples)
        self.classifier = nn.Linear(flatten_dim, n_classes)

    def _infer_flatten_dim(self, n_channels, n_samples):
        with torch.no_grad():
            x = torch.zeros(1, 1, n_channels, n_samples)
            x = torch.cat([branch(x) for branch in self.temporal_branches], dim=1)
            x = self.block1(x)
            x = self.block2(x)
        return x.flatten(start_dim=1).shape[1]

    def forward(self, x):
        x = torch.cat([branch(x) for branch in self.temporal_branches], dim=1)
        x = self.block1(x)
        x = self.block2(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)


model = EEGNetInception(n_channels=N_CHANNELS, n_samples=N_SAMPLES)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'EEGNetInception params: {n_params:,}')

EEGNetInception params: 3,798


/Users/tedsoderberg/Desktop/Duke/Academic/Semester 8/CS590/eeg-cs590/eeg_venv/lib/python3.13/site-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv2d(


## 5. Training

In [15]:
def train(model, train_loader, val_loader, epochs=50, lr=1e-3):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        # ── Train ────────────────────────────────────────────────────────────
        model.train()
        total_loss = 0.0
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # ── Validate ─────────────────────────────────────────────────────────
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                correct += (model(X).argmax(1) == y).sum().item()
                total   += len(y)

        print(f'Epoch {epoch:3d} | Loss: {total_loss:.2f} | Val Acc: {correct/total:.4f}')

In [ ]:
train(model, train_loader, val_loader, epochs=50)

Epoch   1 | Loss: 55.44 | Val Acc: 0.8198
Epoch   2 | Loss: 45.61 | Val Acc: 0.8232
Epoch   3 | Loss: 43.69 | Val Acc: 0.8306
Epoch   4 | Loss: 42.14 | Val Acc: 0.8455
Epoch   5 | Loss: 41.45 | Val Acc: 0.8340
Epoch   6 | Loss: 40.68 | Val Acc: 0.8347
Epoch   7 | Loss: 40.29 | Val Acc: 0.8381
Epoch   8 | Loss: 39.53 | Val Acc: 0.8360
Epoch   9 | Loss: 39.49 | Val Acc: 0.8408
Epoch  10 | Loss: 39.01 | Val Acc: 0.8367
Epoch  11 | Loss: 39.12 | Val Acc: 0.8245
Epoch  12 | Loss: 38.24 | Val Acc: 0.8388


## 6. Test Evaluation

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            correct += (model(X).argmax(1) == y).sum().item()
            total   += len(y)
    return correct / total

test_acc = evaluate(model, test_loader)
print(f'Test Accuracy: {test_acc:.4f}')

Test Accuracy: 0.8312
